## Select Valid (all not nan) Column Associated with Kelvin Wave Events

### Import package

In [1]:
# Limit numerical libraries and scikit-learn to one CPU thread
import os

for variable in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
    "BLIS_NUM_THREADS",
):
    os.environ[variable] = "1"


import numpy as np
import netCDF4 as nc

from sklearn.linear_model import Ridge

from matplotlib import pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import os

### Acquire linear response function

#### Load data

In [2]:
# Load significant events
time_idx: np.ndarray = np.loadtxt(
    "/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/significant_time_idx.txt"
)

lon_val: np.ndarray = np.loadtxt(
    "/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/significant_lon_value.txt"
)

# Load data with removing first 3 harmonics
with nc.Dataset("/data92/b11209013/ERA5_GRIB/Data/tropical_-10_10/w.nc", "r") as w_ds:

    era5_coords: dict[str, np.ndarray] = {
        key: w_ds.variables[key][...]
        for key in w_ds.dimensions.keys()
    }

    lat_lim = (era5_coords["lat"] >= -5.0) & (era5_coords["lat"] <= 5.0)

    w: np.ndarray = w_ds.variables["w"][..., lat_lim, :].astype(np.float32)

# Load CloudSat
with nc.Dataset("/data92/b11209013/CloudSat/DATA/QR_gridded_15layer_-5_5.nc", "r") as cs_ds:

    LW: np.ndarray = cs_ds.variables["QLW"][...].astype(np.float32)
    SW: np.ndarray = cs_ds.variables["QSW"][...].astype(np.float32)

#### Preprocessing

In [3]:
# remove time and zonal mean
w_anom : np.ndarray = w  - np.nanmean(w, axis=(0, -1), keepdims=True)
LW_anom: np.ndarray = LW - np.nanmean(LW, axis=(0, -1), keepdims=True)
SW_anom: np.ndarray = SW - np.nanmean(SW, axis=(0, -1), keepdims=True)

del w, LW, SW

#### Select time and longitude based on OLR-based events

In [4]:
# convert OLR-based longitude values to longitude index
lon_idx: np.ndarray = np.array([
    np.argmin(np.abs(era5_coords["lon"] - l))
    for l in lon_val
])

x_half = w_anom.shape[-1]//2

# select time slice and centering
w_sel, LW_sel, SW_sel = [], [], []

for (i, ti) in enumerate(time_idx):

    ti = int(ti)

    w_sel.append(np.roll(w_anom[ti], shift=x_half - lon_idx[i], axis=-1))
    LW_sel.append(np.roll(LW_anom[ti], shift=x_half - lon_idx[i], axis=-1))
    SW_sel.append(np.roll(SW_anom[ti], shift=x_half - lon_idx[i], axis=-1))

# Convert selected snapshots into array
w_sel_array : np.ndarray = np.array(w_sel).transpose(1, 3, 0, 2) # shape: (nz, nx, nt, ny)
LW_sel_array: np.ndarray = np.array(LW_sel).transpose(1, 3, 0, 2) # shape: (nz, nx, nt, ny)
SW_sel_array: np.ndarray = np.array(SW_sel).transpose(1, 3, 0, 2) # shape: (nz, nx, nt, ny)

del w_sel, LW_sel, SW_sel

#### Find valid column

In [5]:
# masking
LW_finite = np.isfinite(np.ma.filled(LW_sel_array, np.nan))
SW_finite = np.isfinite(np.ma.filled(SW_sel_array, np.nan))

valid_mask = np.all(LW_finite & SW_finite, axis=0)

# filter for the three arrays
w_valid : np.ndarray = w_sel_array[:, valid_mask]
LW_valid: np.ndarray = LW_sel_array[:, valid_mask]
SW_valid: np.ndarray = SW_sel_array[:, valid_mask]

#### Find Jacobian Matrixbetween LW-w, SW-w relations

In [6]:
# apply Ridge regression
LW_model: Ridge = Ridge(alpha=1, solver="svd")
LW_model.fit(w_valid.T, LW_valid.T)

SW_model: Ridge = Ridge(alpha=1, solver="svd")
SW_model.fit(w_valid.T, SW_valid.T)

J_LW = LW_model.coef_
J_SW = SW_model.coef_

#### Save Jacobain and associated files

In [7]:
np.savetxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/Jacob_LW.txt", J_LW)
np.savetxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/Jacob_SW.txt", J_SW)

np.savetxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/plev.txt", era5_coords["plev"])

### Use composite for verification

#### Load composite fields

In [8]:
# Load LW composite profile
LW_composite: np.ndarray = np.loadtxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/LW_composite.txt")

# Load SW composite profile
SW_composite: np.ndarray = np.loadtxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/SW_composite.txt")

# Load w composite profile
w_composite: np.ndarray = np.loadtxt("/home/b11209013/KW_CloudSat/Files/ERA5_GRIB/w_composite.txt")

#### Calculate prediction with composite vertical motion

In [9]:
LW_prediction: np.ndarray = J_LW @ w_composite

SW_prediction: np.ndarray = J_SW @ w_composite

#### Visualizing the composite and prediction profiles of LW and SW

In [10]:
# Relative longitude after centering each event at the middle grid point
dlon = float(np.abs(np.diff(era5_coords["lon"][:2]))[0])
relative_lon = (np.arange(LW_prediction.shape[-1]) - LW_prediction.shape[-1] // 2) * dlon
pressure_hpa = era5_coords["plev"] / 100.0
lon_window = (relative_lon >= -50.0) & (relative_lon <= 50.0)

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 14,
        "axes.labelsize": 14,
        "axes.titlesize": 16,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "legend.fontsize": 12,
        "figure.titlesize": 18,
    }
)

def plot_prediction_verification(prediction, composite, component):
    """Plot a longitude-pressure comparison and its -50 to 50 degree mean."""
    # Use one symmetric scale for the shading and contours so their
    # magnitudes can be compared directly within the displayed window.
    amplitude = np.nanmax(
        np.abs(np.concatenate((prediction[:, lon_window], composite[:, lon_window])))
    )
    levels = np.linspace(-amplitude, amplitude, 11)
    contour_levels = levels

    fig, (ax_profile, ax_mean) = plt.subplots(
        1, 2, figsize=(12, 5), sharey=True,
        gridspec_kw={"width_ratios": (4.2, 1.25), "wspace": 0.08},
        constrained_layout=True,
    )

    shading = ax_profile.contourf(
        relative_lon, pressure_hpa, prediction,
        levels=levels, cmap="RdBu_r", extend="both",
    )
    contours = ax_profile.contour(
        relative_lon[::3], pressure_hpa, composite[:, ::3],
        levels=contour_levels, colors="k", linewidths=0.8,
    )
    ax_profile.clabel(contours, fontsize=7, fmt="%.2f", inline=True)
    ax_profile.axvline(0.0, color="0.35", lw=0.8, ls=":")
    ax_profile.set_xlim(-50.0, 50.0)
    ax_profile.set_ylim(pressure_hpa.max(), pressure_hpa.min())
    ax_profile.set_xlabel("Relative longitude (degrees)")
    ax_profile.set_ylabel("Pressure (hPa)")
    ax_profile.set_title(f"{component}: prediction (shading), composite (contours)")

    prediction_mean = np.nanmean(prediction[:, lon_window], axis=1)
    composite_mean = np.nanmean(composite[:, lon_window], axis=1)
    ax_mean.plot(prediction_mean, pressure_hpa, color="tab:red", lw=2, label="Prediction")
    ax_mean.plot(composite_mean, pressure_hpa, color="k", lw=2, ls="--", label="Composite")
    ax_mean.fill_betweenx(
        pressure_hpa, prediction_mean, composite_mean, color="0.7", alpha=0.3,
        label="Difference",
    )
    ax_mean.axvline(0.0, color="0.5", lw=0.8)
    ax_mean.set_xlabel("Mean anomaly")
    ax_mean.set_title("Mean: -50° to 50°")
    ax_mean.grid(alpha=0.25)
    ax_mean.legend(loc="best", fontsize=8)

    colorbar = fig.colorbar(shading, ax=ax_profile, pad=0.02)
    colorbar.set_label("Predicted heating-rate anomaly")
    plt.savefig(f"/home/b11209013/KW_CloudSat/Figure/ERA5_GRIB/{component}.png", dpi=300, bbox_inches="tight")
    plt.close()


plot_prediction_verification(LW_prediction, LW_composite, "Longwave")
plot_prediction_verification(SW_prediction, SW_composite, "Shortwave")

In [11]:
# Visualize the LW and SW Jacobians: rows are heating-rate response levels,
# and columns are the pressure levels at which vertical motion is applied.
pressure_hpa = era5_coords["plev"] / 100.0

fig, axes = plt.subplots(
    1, 2, figsize=(14, 5.5), sharey=True, constrained_layout=True
)

for ax, jacobian, component in zip(
    axes, (J_LW, J_SW), ("Longwave", "Shortwave")
):
    limit = np.nanmax(np.abs(jacobian))
    norm = TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)

    mesh = ax.pcolormesh(
        pressure_hpa, pressure_hpa, jacobian,
        cmap="RdBu_r", norm=norm, shading="nearest",
    )
    ax.plot(pressure_hpa, pressure_hpa, color="0.25", ls=":", lw=1)
    ax.set_xlim(pressure_hpa.max(), pressure_hpa.min())
    ax.set_ylim(pressure_hpa.max(), pressure_hpa.min())
    ax.set_xlabel(r"Input $w$ pressure (hPa)")
    ax.set_title(f"{component}")

    colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
    colorbar.set_label(r"$\partial Q / \partial w$")

axes[0].set_ylabel("Output heating-rate pressure (hPa)")
fig.suptitle("Radiative-heating response to vertical motion", fontsize=18)
plt.savefig("/home/b11209013/KW_CloudSat/Figure/ERA5_GRIB/Jacobian.png", dpi=300, bbox_inches="tight")
plt.close()